In [198]:
import sys
from pathlib import Path
import time

from scipy.optimize import minimize

import pprint
from matplotlib import font_manager
import matplotlib.pyplot as plt
# Add the path to the downloaded fonts
font_dirs = ['/home/simoneponcioni/Documents/99_OTHERS/my_fonts/']  # Replace with the actual path to your fonts
font_files = font_manager.findSystemFonts(fontpaths=font_dirs)

for font_file in font_files:
    font_manager.fontManager.addfont(font_file)

# Load the style file
plt.style.use('../../01_CODE/src/style/paper-style.mplstyle')

# Set up fonts
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Outfit', 'Cabin']

sys.path.append(str(Path('/home/simoneponcioni/Documents/01_PHD/03_Methods/HR-pQCT_database/01_CODE/src')))
import statistics_hrpqct as statistics_hrpqct
import dataclasses_hrpqct as dataclass_hrpqct
import pandas as pd
from sklearn.linear_model import LinearRegression
import numpy as np
from scipy.stats import ttest_ind

In [199]:
# Import raw datafile from csv raw filepath#
dir_path = Path('../../00_DB/')
df = Path('HR-pQCT_database_expanded_2025-06-16_16-18.csv')
dataset_path = dir_path / df
output_path = f'{df.name}_quad_tai.csv'

In [200]:
# Instantiate class dataclass_hrpqct class for the database#
name_combined = 'Combined'
name_originator = 'POS'

common_df = pd.read_csv(dataset_path, sep=',')
dataset = dataclass_hrpqct.HRpQCT_Dataset(df=common_df, hfe_expansion=True)

In [201]:
# * This one includes the 6 radar's properties and the ones recommended by Whittier et al. (2020)

Tibia_total_volumetric_bone_mineral_density_g_HA_ccm = dataset.Tibia_total_volumetric_bone_mineral_density_mg_HA_ccm * 1e-3
Tibia_cortical_vBMD_g_HA_ccm = dataset.Tibia_cortical_vBMD_mg_HA_ccm * 1e-3
# Tibia_maximum_force_at_failure_kN = dataset.Tibia_maximum_force_at_failure_N * 1e-3
Tibia_maximum_force_at_failure_kN = dataset.Tibia_yield_force * 1e-3
Tibia_trabecular_vBMD_g_HA_ccm = dataset.Tibia_trabecular_vBMD_mg_HA_ccm * 1e-3

Radius_total_volumetric_bone_mineral_density_g_HA_ccm = dataset.Radius_total_volumetric_bone_mineral_density_mg_HA_ccm * 1e-3
Radius_cortical_vBMD_g_HA_ccm = dataset.Radius_cortical_vBMD_mg_HA_ccm * 1e-3
# Radius_maximum_force_at_failure_kN = dataset.Radius_maximum_force_at_failure_N * 1e-3
Radius_maximum_force_at_failure_kN = dataset.Radius_yield_force * 1e-3
Radius_trabecular_vBMD_g_HA_ccm = dataset.Radius_trabecular_vBMD_mg_HA_ccm * 1e-3


dataset_properties = [
    dataset.dexa_femoral_neck_BMD_g_cm2,
    Tibia_total_volumetric_bone_mineral_density_g_HA_ccm,
    Tibia_cortical_vBMD_g_HA_ccm,
    dataset.Tibia_trabecular_bone_volume_fraction,
    dataset.Tibia_rel_cortical_thickness,
    dataset.Tibia_yield_stress,
    dataset.Tibia_da,
    Tibia_maximum_force_at_failure_kN,
    dataset.Tibia_cortical_thickness_mm,
    dataset.Tibia_cortical_porosity,
    dataset.Tibia_trabecular_number_1_mm,
    dataset.Tibia_trabecular_separation_mm,
    dataset.Tibia_trabecular_thickness_mm,
    Tibia_trabecular_vBMD_g_HA_ccm,
    
    Radius_total_volumetric_bone_mineral_density_g_HA_ccm,
    Radius_cortical_vBMD_g_HA_ccm,
    dataset.Radius_trabecular_bone_volume_fraction,
    dataset.Radius_rel_cortical_thickness,
    dataset.Radius_yield_stress,
    dataset.Radius_da,
    Radius_maximum_force_at_failure_kN,
    dataset.Radius_cortical_thickness_mm,
    dataset.Radius_cortical_porosity,
    Radius_trabecular_vBMD_g_HA_ccm,
    dataset.Radius_trabecular_number_1_mm,
    dataset.Radius_trabecular_separation_mm,
    dataset.Radius_trabecular_thickness_mm,
]


In [202]:
# precision errors in % (implemented in the dataclass as stpe)
dataset.dexa_femoral_neck_BMD_g_cm2.stpe = 1.2
Tibia_total_volumetric_bone_mineral_density_g_HA_ccm.stpe = 0.427
Tibia_cortical_vBMD_g_HA_ccm.stpe = 0.37
dataset.Tibia_trabecular_bone_volume_fraction.stpe = 0.654
dataset.Tibia_rel_cortical_thickness.stpe = 1.315
dataset.Tibia_yield_stress.stpe = 3.514
dataset.Tibia_da.stpe = 1.582
Tibia_maximum_force_at_failure_kN.stpe = 4.13
dataset.Tibia_cortical_thickness_mm.stpe = 1.388
dataset.Tibia_cortical_porosity.stpe = 5.176
Tibia_trabecular_vBMD_g_HA_ccm.stpe = 0.509
dataset.Tibia_trabecular_number_1_mm.stpe = 2.027
dataset.Tibia_trabecular_separation_mm.stpe = 1.561
dataset.Tibia_trabecular_thickness_mm.stpe = 0.605

Radius_total_volumetric_bone_mineral_density_g_HA_ccm.stpe = 0.591
Radius_cortical_vBMD_g_HA_ccm.stpe = 1.247
dataset.Radius_trabecular_bone_volume_fraction.stpe = 3.7
dataset.Radius_trabecular_to_cortical_bmc.stpe = 1e-6
dataset.Radius_rel_cortical_thickness.stpe = 3.21
dataset.Radius_yield_stress.stpe = 5.71
dataset.Radius_da.stpe = 1.567
Radius_maximum_force_at_failure_kN.stpe = 4.62
dataset.Radius_cortical_thickness_mm.stpe = 1.843
dataset.Radius_cortical_porosity.stpe = 11.262
Radius_trabecular_vBMD_g_HA_ccm.stpe = 3.314
dataset.Radius_trabecular_number_1_mm.stpe = 2.799
dataset.Radius_trabecular_separation_mm.stpe = 2.866
dataset.Radius_trabecular_thickness_mm.stpe = 1.96

In [203]:
# Custom formatting function
def custom_format(x):
    if abs(x) > 1000 or (abs(x) < 0.001 and x != 0):
        return f'{x:.3f}'
    else:
        return f'{x:.3f}'

def compute_gluer_tai(pe, median_change_percent_per_year):
    """
    Calculate TAI using: TAI = (1.8 × PE_st) / (median change per annum %)
    
    Args:
        pe: precision error (PE_st) as a percentage
        median_change_percent_per_year: median change per annum as percentage per year
    
    Returns:
        TAI value in years
    """
    if abs(median_change_percent_per_year) < 1e-10:  # Avoid division by zero
        return float('inf')
    
    tai_years = (pe * 1.8) / abs(median_change_percent_per_year)
    return tai_years

In [204]:
# Initialize list to store all results
all_results = []

# Initialize results dictionaries for males and females (keep for LaTeX)
results_dict_male = {}
results_dict_female = {}

for prop in dataset_properties:
    results_dict_male[prop.name] = {}
    results_dict_female[prop.name] = {}
    
    # Determine site from property name
    if 'Tibia' in prop.name:
        site = 'Tibia'
    elif 'Radius' in prop.name:
        site = 'Radius'
    else:
        site = 'DEXA'  # For femoral neck BMD
    
    for sex in (20, 25):
        # Extract only dataclass for one patient, it will determine which sex is the patient for further analysis
        patient_UID = sex
        df_patient = common_df[common_df["UID"] == patient_UID]
        dataset_patient = dataclass_hrpqct.HRpQCT_Dataset(df=df_patient)
        gender = dataset_patient.df.Gender.values[0]
        
        db_stats = statistics_hrpqct.Statistics(
            df=dataset.df, 
            df_patient=dataset_patient.df, 
            name='HRpQCT_common_database', 
            originator='POS', 
            savefig=False, 
            showfig=True
        )

        # Calculate intercepts and slopes
        # returns (intercept, slope)
        intercepts_slopes = db_stats.linear_regression_constraint_paper(dataset.Age, prop, dataset.__avg_std__(prop, gender))
        
        # Extract average values from __avg_std__
        avg_ref, std_ref = dataset.__avg_std__(prop, gender)
        
        # Calculate median regression slope
        res = db_stats.median_regression_above_37(dataset.Age, prop)
        
        # Get median slope (absolute change per year)
        median_slope = res.params["x"]

        # Convert to percentage change per year
        # This is what the paper calls "median change per annum"
        median_change_per_annum_percent = (median_slope / avg_ref) * 100
        
        # Calculate TAI using the correct formula
        tai_gluer = compute_gluer_tai(prop.stpe, median_change_per_annum_percent)
        
        result_row = {
            'Property': prop.name,
            'Site': site,
            'Gender': gender,
            'Mean': np.round(avg_ref, 3),
            'Std': np.round(std_ref, 3),
            'CV': np.round(std_ref / avg_ref, 3),
            'PE_st_percent': np.round(prop.stpe, 3),
            'Median_slope_absolute': np.round(median_slope, 6),
            'Median_change_percent_per_year': np.round(median_change_per_annum_percent, 3),
            'TAI_years': np.round(tai_gluer, 3),
            'Linear_slope': np.round(intercepts_slopes[1], 6),
            'Linear_change_percent_per_year': np.round((intercepts_slopes[1] / avg_ref) * 100, 3)
        }
        all_results.append(result_row)
        
        # Store results in the appropriate dictionary (keep for LaTeX)
        if gender == 'Male':
            results_dict_male[prop.name]['$\\mu$'] = np.round(avg_ref, 3)
            results_dict_male[prop.name]['$\\sigma$'] = np.round(std_ref, 3)
            results_dict_male[prop.name]['$\\frac{\\sigma}{\\mu}$'] = np.round(std_ref / avg_ref, 3)
            results_dict_male[prop.name]['$\\mathrm{PE_{st}} [\\%]$'] = np.round(prop.stpe, 3)
            # results_dict_male[prop.name]['$\\Delta\\mu/y$'] = np.round(intercepts_slopes[1], 3)
            # results_dict_male[prop.name]['$\\frac{\\Delta\\mu}{\\mu}/year$ [\\%]'] = np.round((intercepts_slopes[1] / avg_ref) * 100, 3)
            results_dict_male[prop.name]['{$\frac{\Delta\tilde{\mu}}{\tilde{\mu}}/y$ [\%]}'] = np.round(median_change_per_annum_percent, 3)
            # results_dict_male[prop.name]['$\\frac{\\Delta\\mu}{SE} [\\%]$'] = np.round(intercepts_slopes[1] / stderr_over37, 3)
            results_dict_male[prop.name]['$\\mathrm{TAI_{p}}$ [\\%]'] = np.round(tai_gluer, 3)
        else:
            results_dict_female[prop.name]['$\\mu$'] = np.round(avg_ref, 3)
            results_dict_female[prop.name]['$\\sigma$'] = np.round(std_ref, 3)
            results_dict_female[prop.name]['$\\frac{\\sigma}{\\mu}$'] = np.round(std_ref / avg_ref, 3)
            results_dict_female[prop.name]['$\\mathrm{PE_{st}} [\\%]$'] = np.round(prop.stpe, 3)
            # results_dict_female[prop.name]['$\\Delta\\mu/y$'] = np.round(intercepts_slopes[1], 3)
            # results_dict_female[prop.name]['$\\frac{\\Delta\\mu}{\\mu}/year$ [\\%]'] = np.round((intercepts_slopes[1] / avg_ref) * 100, 3)
            # results_dict_female[prop.name]['$\\frac{\\Delta\\tilde{\\mu}}{\\tilde{\\mu}}/year$ [\\%]'] = np.round(median_change_per_annum_percent, 3),
            results_dict_female[prop.name]['{$\frac{\Delta\tilde{\mu}}{\tilde{\mu}}/y$ [\%]}'] = np.round(median_change_per_annum_percent, 3)
            # results_dict_female[prop.name]['$\\frac{\\Delta\\mu}{SE} [\\%]$'] = np.round(intercepts_slopes[1] / stderr_over37, 3)
            results_dict_female[prop.name]['$\\mathrm{TAI_{p}}$ [\\%]'] = np.round(tai_gluer, 3)

# Convert results dictionaries to pandas DataFrames (keep for LaTeX)
results_df_male = pd.DataFrame.from_dict(results_dict_male, orient='index')
results_df_female = pd.DataFrame.from_dict(results_dict_female, orient='index')

# Apply the desired style
styled_df_male = results_df_male.style.format({
    '$\\mu$': custom_format,
    '$\\sigma$': custom_format,
    '$\\frac{\\sigma}{\\mu}$': custom_format,
    # 'SE': custom_format,
    '$\\mathrm{PE_{st}} [\\%]$': custom_format,
    '$\\Delta\\mu$': custom_format,
    # '$\\frac{\\Delta\\mu}{\\mu}/year$ [\\%]': custom_format,
    '{$\frac{\Delta\tilde{\mu}}{\tilde{\mu}}/y$ [\%]}': custom_format,
    # '$\\frac{\\Delta\\mu}{SE} [\\%]$': custom_format,
    '$\\mathrm{TAI_{p}}$ [\\%]': custom_format,
})

styled_df_female = results_df_female.style.format({
    '$\\mu$': custom_format,
    '$\\sigma$': custom_format,
    '$\\frac{\\sigma}{\\mu}$': custom_format,
    # 'SE': custom_format,
    '$\\mathrm{PE_{st}} [\\%]$': custom_format,
    '$\\Delta\\mu$': custom_format,
    # '$\\frac{\\Delta\\mu}{\\mu}/year$ [\\%]': custom_format,
    '{$\frac{\Delta\tilde{\mu}}{\tilde{\mu}}/y$ [\%]}': custom_format,
    # '$\\frac{\\Delta\\mu}{SE} [\\%]$': custom_format,
    '$\\mathrm{TAI_{p}}$ [\\%]': custom_format,
})

# Convert to LaTeX format
caption_male = "Table showing the descriptive statistics for the male cohort. \\textit{\\(\\mu\\): population mean, \\(\\sigma\\): standard deviation, \\(\\frac{\\sigma}{\\mu}\\): coefficient of variation, \\(\\mathrm{PE}_{st}\\): measurement precision, \\(\\frac{\\Delta\\mu}{\\mu}/year\\): relative rate of change of the mean per year, \\(\\mathrm{TAI_{st}}\\): short-term trend assessment interval}."
caption_female = "Descriptive statistics for the female cohort. \\textit{\\(\\mu\\): population mean, \\(\\sigma\\): standard deviation, \\(\\frac{\\sigma}{\\mu}\\): coefficient of variation, \\(\\mathrm{PE}_{st}\\): measurement precision, \\(\\frac{\\Delta\\mu}{\\mu}/year\\): relative rate of change of the mean per year, \\(\\mathrm{TAI_{st}}\\): short-term trend assessment interval}."

filename_male = f'descriptive_statistics_male_quad.tex'
filename_female = f'descriptive_statistics_female_quad.tex'
latex_table_male = styled_df_male.to_latex(column_format="lrrrrrrrr",
                                        position="h",
                                        position_float="centering",
                                        hrules=True,
                                        label="tab:desc_stats_male",
                                        caption=caption_male,
                                        multirow_align="t",
                                        multicol_align="r",
                                        clines=None,
                                        buf=filename_male)

latex_table_female = styled_df_female.to_latex(column_format="lrrrrrrrr",
                                            position="h",
                                            position_float="centering",
                                            hrules=True,
                                            label="tab:desc_stats_female",
                                            caption=caption_female,
                                            multirow_align="t",
                                            multicol_align="r",
                                            clines=None,
                                            buf=filename_female)

print(latex_table_male)
print(latex_table_female)

None
None


In [206]:
# Define the filenames
filenames = ['descriptive_statistics_female_quad.tex', 'descriptive_statistics_male_quad.tex']

# Dictionary mapping old names to new formatted names
replacements = {
    "Tot.vBMD [mg HA/cmm]": "* \\acrshort{ttvbmd} [\\gHAccm]",
    "Ct.vBMD [mg HA/cmm]": "* \\acrshort{ctvbmd} [\\gHAccm]",
    "Tb.BV/TV [1]": "* \\acrshort{tb_bvtv} [-]",
    "Relative cortical thickness [-]": "* \\acrshort{relctth} [-]",
    "Tb.DA" : "* \\acrshort{tb_da} [-]",
    "y.Stress [MPa]": "* \\acrshort{app_ys} [MPa]",
    "y.Force [N]": "\\(^\\filledtriangleup\\) \\acrshort{fmax} [kN]",
    "Ct.Th [mm]": "\\(^\\filledtriangleup\\) \\acrshort{ctth} [mm]",
    "Ct.Po [1]": "\\(^\\filledtriangleup\\) \\acrshort{ctpo} [-]",
    "Tb.vBMD [mg HA/cmm]": "\\(^\\filledtriangleup\\) \\acrshort{tb_vbmd} [\\gHAccm]",
    "Tb.N [1/mm]": "\\(^\\filledtriangleup\\) \\acrshort{tbn} [1/mm]",
    "Tb.Sp [mm]": "\\(^\\filledtriangleup\\) \\acrshort{tbsp} [mm]",
    "Tb.Th [mm]": "\\(^\\filledtriangleup\\) \\acrshort{tbth} [mm]",
}

# Loop through each file
for file in filenames:
    with open(file, 'r') as f:
        content = f.read()
    
    # Line-by-line processing
    lines = content.split('\n')
    new_lines = []
    corrected_ok_t = 0
    corrected_ok_r = 0
    
    for line in lines:
        if line.startswith("Tibia: "):
            if corrected_ok_t == 0:
                new_lines.append("Tibia &&&&&&& \\\\")
                corrected_ok_t += 1
            new_lines.append(line.replace("Tibia: ", "\\quad "))
        elif line.startswith("Radius: "):
            if corrected_ok_r == 0:
                new_lines.append("Radius &&&&&&& \\\\")
                corrected_ok_r += 1
            new_lines.append(line.replace("Radius: ", "\\quad "))
        else:
            # Apply replacements
            for old, new in replacements.items():
                if old in line:
                    print('Updating line:', line)
                    line = line.replace(old, new)
            new_lines.append(line)
    
    # Save updated content
    modified_content = '\n'.join(new_lines)
    with open(file, 'w') as f:
        f.write(modified_content)


Updating line: \quad Tot.vBMD [mg HA/cmm] & 0.266 & 0.036 & 0.135 & 0.427 & -0.753 & 1.021 \\
Updating line: \quad Ct.vBMD [mg HA/cmm] & 0.865 & 0.032 & 0.037 & 0.370 & -0.657 & 1.013 \\
Updating line: \quad Tb.BV/TV [1] & 0.286 & 0.044 & 0.153 & 0.654 & -0.367 & 3.207 \\
Updating line: \quad Relative cortical thickness [-] & 0.065 & 0.012 & 0.179 & 1.315 & -1.024 & 2.312 \\
Updating line: \quad y.Stress [MPa] & 10.988 & 2.279 & 0.207 & 3.514 & -1.368 & 4.625 \\
Updating line: \quad Tb.DA & 1.726 & 0.071 & 0.041 & 1.582 & -0.192 & 14.822 \\
Updating line: \quad y.Force [N] & 9.967 & 2.187 & 0.219 & 4.130 & -0.872 & 8.521 \\
Updating line: \quad Ct.Th [mm] & 1.096 & 0.159 & 0.145 & 1.388 & -0.874 & 2.860 \\
Updating line: \quad Ct.Po [1] & 0.015 & 0.006 & 0.427 & 5.176 & 1.142 & 8.158 \\
Updating line: \quad Tb.N [1/mm] & 1.590 & 0.155 & 0.098 & 2.027 & 0.197 & 18.518 \\
Updating line: \quad Tb.Sp [mm] & 0.591 & 0.069 & 0.116 & 1.561 & -0.078 & 36.215 \\
Updating line: \quad Tb.Th [mm] 